# 07. Frozen-model external evaluation — separate ElasticNet and XGBoost runs

This paper-ready notebook applies the **prespecified** CCLE-trained models to all 20 external cohorts without external model selection or refitting.

The two multigene families are deliberately executed in **separate cells**:

1. prepare/reuse the shared disk-backed validation workspace;
2. run or resume **ElasticNet only**, with a live `tqdm` protein progress bar;
3. run or resume **XGBoost only**, with an independent progress bar;
4. finalize the portable cohort checkpoints and compact metrics only after both inventories are complete.

The same memory-safe design is retained: the external matrices and prediction matrices remain disk-backed, one HDF5 model is loaded at a time, predictions are generated in bounded sample batches, and no long sample-by-protein table is materialized. Progress is checkpointed to `state.json`, so interrupting either model family does not discard completed proteins.

**Run it top to bottom.** Proteins that failed in an earlier run are re-queued automatically, and the saved-normalizer check is reported but does not block predictions (section 2b).

In [1]:
# >>> Run from this notebook's own folder (05. validation/notebooks) so every relative path works
import os
from pathlib import Path

STAGE, SUB = "05. validation", "notebooks"
_cwd = Path.cwd().resolve()
_nb_dir = next((c for base in [_cwd, *_cwd.parents]
                for c in [base, base / SUB, base / STAGE / SUB]
                if c.name == SUB and c.parent.name == STAGE and (c / "paper_validation.py").is_file()), None)
if _nb_dir is None:
    raise FileNotFoundError(f"Could not locate '{STAGE}/{SUB}' (with paper_validation.py) from {_cwd}")
os.chdir(_nb_dir)
print("Working directory:", Path.cwd())

Working directory: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\notebooks


In [2]:
# >>> Environment check: stops here with the exact fix if a package is broken or missing
import importlib

REQUIRED = ["numpy", "pandas", "scipy", "sklearn", "h5py", "joblib", "tqdm", "xgboost"]
PINNED = {"numpy": "numpy==1.26.4", "pandas": "pandas==2.2.3", "scipy": "scipy==1.11.4",
          "sklearn": "scikit-learn==1.7.2", "h5py": "h5py==3.11.0", "pyarrow": "pyarrow==16.1.0",
          "joblib": "joblib", "tqdm": "tqdm", "xgboost": "xgboost"}
problems = {}
for mod in ["pyarrow"] + REQUIRED:                 # a broken pyarrow also breaks scikit-learn
    try:
        m = importlib.import_module(mod)
        print(f"  OK  {mod:<8} {getattr(m, '__version__', '')}")
    except ModuleNotFoundError:
        if mod == "pyarrow":
            print("  --  pyarrow  not installed (optional; tables are then written as CSV)")
        else:
            problems[mod] = "not installed"
    except Exception as exc:
        problems[mod] = f"{type(exc).__name__}: {exc}"
if problems:
    for mod, msg in problems.items():
        print(f"  !!  {mod}: {msg}")
    print("\nFix: run the line below, RESTART THE KERNEL, then run this notebook again:")
    print('%pip install --no-cache-dir "numpy==1.26.4" ' + " ".join(f'"{PINNED.get(m, m)}"' for m in problems))
    raise RuntimeError("Environment not ready - see the fix printed above.")
print("Environment OK")

  OK  pyarrow  11.0.0
  OK  numpy    1.26.4
  OK  pandas   2.3.2
  OK  scipy    1.16.3
  OK  sklearn  1.7.2
  OK  h5py     3.11.0
  OK  joblib   1.2.0
  OK  tqdm     4.65.0
  OK  xgboost  2.1.3
Environment OK


In [3]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "7_inference_all_models"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

Figures are saved to C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\notebooks\figures_pdf\7_inference_all_models


In [4]:
from pathlib import Path
import gc
import json
import os
import sys

# Bound BLAS/OpenMP/XGBoost threading before importing numerical libraries.
for _thread_var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"
):
    os.environ.setdefault(_thread_var, "1")

# Find paper_validation.py beside the notebook/package or through an explicit override.
search_roots = []
if os.environ.get("PAPER_VALIDATION_MODULE_DIR"):
    search_roots.append(Path(os.environ["PAPER_VALIDATION_MODULE_DIR"]).expanduser().resolve())
cwd = Path.cwd().resolve()
for base in [cwd, *cwd.parents]:
    for candidate in [base, base / "paper_ready_external_validation", base / "notebooks", base / "notebooks2"]:
        if candidate not in search_roots:
            search_roots.append(candidate)
for candidate in search_roots:
    if (candidate / "paper_validation.py").exists():
        sys.path.insert(0, str(candidate))
        MODULE_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "paper_validation.py was not found. Keep the updated module beside this notebook "
        "or set PAPER_VALIDATION_MODULE_DIR."
    )

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
import paper_validation as pv

PROJECT_ROOT = pv.detect_project_root()
RUN_ROOT = Path(os.environ.get(
    "PAPER_VALIDATION_RUN_ROOT",
    PROJECT_ROOT / "results_external_validation_paper_v1",
)).expanduser().resolve()
LAYOUT = pv.make_output_layout(RUN_ROOT)
MANIFEST_PATH = RUN_ROOT / "validation_manifest.json"
SMOKE_TEST = os.environ.get("PAPER_VALIDATION_SMOKE_TEST", "0") == "1"

print("Notebook 07 — separate model-family execution")
print("Shared module:", MODULE_DIR / "paper_validation.py")
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Smoke-test mode:", SMOKE_TEST)

Notebook 07 — separate model-family execution
Shared module: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\notebooks\paper_validation.py
Project root: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation
Run root: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\results_external_validation_paper_v1
Smoke-test mode: False


In [5]:
# >>> Robust model loading (no change to inputs or outputs)
# Artifacts pickled with NumPy 2.x load in NumPy 1.x (and vice versa); also installed into pv,
# so the evaluation functions below use it even with an older paper_validation.py.
import io, lzma, pickle, warnings


class _NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except (ModuleNotFoundError, AttributeError):
            if module.startswith("numpy._core"):
                return super().find_class("numpy.core" + module[len("numpy._core"):], name)
            if module.startswith("numpy.core"):
                return super().find_class("numpy._core" + module[len("numpy.core"):], name)
            raise


def _load_h5_artifact_compat(path):
    import h5py
    with h5py.File(path, "r") as h5:
        payload = np.asarray(h5["model/artifact_pickle_xz"][:], dtype=np.uint8)
    artifact = _NumpyCompatUnpickler(io.BytesIO(lzma.decompress(payload.tobytes()))).load()
    artifact.setdefault("normalizer", {"method": "none", "columns": list(map(str, artifact["selected_features"]))})
    return artifact


pv.load_h5_artifact = _load_h5_artifact_compat
# XGBoost warns once per model when the pickle came from an older XGBoost; predictions are unaffected.
warnings.filterwarnings("ignore", message=".*If you are loading a serialized model.*", category=UserWarning)
print("Compatible model loader installed (NumPy", np.__version__ + ")")

Compatible model loader installed (NumPy 1.26.4)


## 1. Load and validate the frozen manifest

Notebook 06 must have completed successfully. This cell verifies the frozen reference, proxy parameters, and exact model inventory before any external prediction is made.

In [6]:
manifest = pv.load_manifest(MANIFEST_PATH)
MANIFEST_HASH = manifest["manifest_hash"]

if manifest["primary_analysis"]["external_model_selection"] is not False:
    raise RuntimeError("The frozen manifest permits external model selection, which is not paper-ready.")
if manifest["primary_analysis"]["external_sctranslator"] != "removed_incomplete":
    raise RuntimeError("External scTranslator must be complete or explicitly removed.")

for label in ["ccle_reference", "linear_proxy_parameters", "model_inventory"]:
    spec = manifest["files"][label]
    path = Path(spec["path"])
    if not path.exists() or pv.file_sha256(path) != spec["sha256"]:
        raise RuntimeError(f"Frozen contract file is missing or changed: {label} -> {path}")

for cohort, spec in manifest["files"]["cohort_contracts"].items():
    path = Path(spec["path"])
    if not path.exists() or pv.file_sha256(path) != spec["sha256"]:
        raise RuntimeError(f"Cohort contract is missing or changed: {cohort} -> {path}")

reference = joblib.load(manifest["files"]["ccle_reference"]["path"])
proxy_parameters = pv.read_table(manifest["files"]["linear_proxy_parameters"]["path"])
model_inventory = pv.read_table(manifest["files"]["model_inventory"]["path"])

inventory_hash = pv.stable_hash(
    model_inventory.sort_values(["display_model", "protein"]).to_dict("records")
)
if inventory_hash != manifest["model_inventory_hash"]:
    raise RuntimeError("The loaded primary model inventory does not match the manifest hash.")

inventory_summary = (
    model_inventory.groupby("display_model", as_index=False)
    .agg(proteins=("protein", "nunique"), files=("path", "size"))
)
print("Manifest hash:", MANIFEST_HASH)
display(inventory_summary)

Manifest hash: 98cdc5451b903c1769428a4757a2859815a411310b36efcb7ddbbf347f2bce50


,display_model,proteins,files
0,ElasticNet,5113,5113
1,XGBoost,5113,5113


## 2. Execution configuration

`RESET_ALL=True` removes the shared evaluation workspace and invalidates previously finalized cohort checkpoints before the ElasticNet cell starts. Leave it `False` to resume.

The progress bar advances **one protein model at a time**. Checkpoints are also flushed periodically, so the bar can be interrupted and resumed without starting from protein 1.

In [7]:
RESET_ALL = os.environ.get("PAPER_VALIDATION_FORCE", "0") == "1"
COMPUTE_SPEARMAN = os.environ.get("PAPER_VALIDATION_SPEARMAN", "1") == "1"
COMPRESSED_NPZ = os.environ.get("PAPER_VALIDATION_COMPRESS_NPZ", "1") == "1"
SAMPLE_BATCH_SIZE = int(os.environ.get(
    "PAPER_VALIDATION_SAMPLE_BATCH", "128" if SMOKE_TEST else "512"
))
CHECKPOINT_EVERY = int(os.environ.get(
    "PAPER_VALIDATION_CHECKPOINT_EVERY", "2" if SMOKE_TEST else "50"
))
# Keep the workspace after finalization by default so either family can be inspected/resumed.
CLEANUP_WORK_AFTER_FINALIZE = os.environ.get("PAPER_VALIDATION_CLEANUP_WORK", "0") == "1"

cohort_bundle_paths = {
    cohort: spec["path"]
    for cohort, spec in manifest["files"]["cohort_contracts"].items()
}
work_dir = LAYOUT["evaluation"] / f"work_{MANIFEST_HASH[:16]}"
state_path = work_dir / "state.json"

def show_family_state():
    if not state_path.exists():
        return pd.DataFrame({
            "display_model": ["ElasticNet", "XGBoost"],
            "completed_proteins": [0, 0],
            "total_proteins": [
                int((model_inventory["display_model"] == "ElasticNet").sum()),
                int((model_inventory["display_model"] == "XGBoost").sum()),
            ],
        })
    state = json.loads(state_path.read_text(encoding="utf-8"))
    rows = []
    for family in ["ElasticNet", "XGBoost"]:
        total = int((model_inventory["display_model"] == family).sum())
        done = int(state.get("progress", {}).get(family, 0))
        rows.append({
            "display_model": family,
            "completed_proteins": done,
            "total_proteins": total,
            "percent": 100.0 * done / total if total else np.nan,
        })
    return pd.DataFrame(rows)

print("RESET_ALL:", RESET_ALL)
print("Sample batch size:", SAMPLE_BATCH_SIZE)
print("Checkpoint every proteins:", CHECKPOINT_EVERY)
print("Work directory:", work_dir)
display(show_family_state())

RESET_ALL: False
Sample batch size: 512
Checkpoint every proteins: 50
Work directory: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\results_external_validation_paper_v1\evaluation\work_98cdc5451b903c17


,display_model,completed_proteins,total_proteins
0,ElasticNet,0,5113
1,XGBoost,0,5113


## 2b. Saved normalizer vs. the CCLE matrix in `data/`

Every ElasticNet and XGBoost model applies **its own saved CCLE-training normalizer** (paper: saved transformations applied without refitting). This cell only *reports* how closely those saved statistics match the CCLE matrix in `05. validation/data/`, which is used for median completion and the same-gene proxies.

Per-model verification is **off by default** (`VERIFY_NORMALIZERS = False`). If the matrix is a slightly different build of the training matrix, as in this project (median gene-mean difference ≈ 0.013 log2(TPM+1)), verification would reject every model without changing any prediction. Set `PAPER_VALIDATION_VERIFY_NORMALIZERS=1` to enforce it, for example once the exact training matrix is in `data/`.

In [8]:
VERIFY_NORMALIZERS = os.environ.get("PAPER_VALIDATION_VERIFY_NORMALIZERS", "0") == "1"

_summary = []
for family in ["ElasticNet", "XGBoost"]:
    inv_f = model_inventory[model_inventory["display_model"] == family].reset_index(drop=True)
    if inv_f.empty:
        continue
    first = pv.load_h5_artifact(inv_f.loc[0, "path"])["normalizer"]
    cols = [str(c) for c in first.get("columns", [])]
    pos = pd.Index(reference["rna_columns"]).get_indexer(cols)
    ok = pos >= 0
    X = reference["rna_train"].iloc[:, pos[ok]].astype(float)
    d_mean = np.abs(X.mean().to_numpy() - np.asarray(first["mean"], float)[ok])
    d_sd = np.abs(X.std(ddof=0).to_numpy() - np.asarray(first["std"], float)[ok])
    # do the models of this family share one normalizer?
    others = [pv.load_h5_artifact(inv_f.loc[i, "path"])["normalizer"] for i in {len(inv_f) // 2, len(inv_f) - 1}]
    shared = all(np.array_equal(np.asarray(o["mean"]), np.asarray(first["mean"])) for o in others)
    _summary.append({"family": family, "method": first.get("method"), "genes": len(cols),
                     "genes_missing_in_CCLE": int((~ok).sum()), "median_abs_dmean": float(np.median(d_mean)),
                     "max_abs_dmean": float(np.max(d_mean)), "median_abs_dsd": float(np.median(d_sd)),
                     "max_abs_dsd": float(np.max(d_sd)), "shared_normalizer_across_models": shared})
normalizer_summary = pd.DataFrame(_summary)
display(normalizer_summary)
pv.write_table(normalizer_summary, LAYOUT["evaluation"] / "saved_normalizer_vs_ccle_reference", index=False)

if normalizer_summary["max_abs_dmean"].max() < 1e-3:
    print("Saved normalizers match the CCLE matrix in data/ (exact training build).")
else:
    print("Saved normalizers differ slightly from the CCLE matrix in data/ (different build of the same data).")
print("Per-model normalizer verification:", "ON" if VERIFY_NORMALIZERS else "OFF (predictions use each model's saved normalizer)")

,family,method,genes,genes_missing_in_CCLE,median_abs_dmean,max_abs_dmean,median_abs_dsd,max_abs_dsd,shared_normalizer_across_models
0,ElasticNet,zscore,22614,0,0.012689,0.194363,0.013291,0.390301,True
1,XGBoost,zscore,22614,0,0.012689,0.194363,0.013291,0.390301,True


Saved normalizers differ slightly from the CCLE matrix in data/ (different build of the same data).
Per-model normalizer verification: OFF (predictions use each model's saved normalizer)


In [9]:
# >>> Re-queue proteins that failed in a previous run, so the family cells below predict them again
if state_path.exists():
    _state = json.loads(state_path.read_text(encoding="utf-8"))
    _requeued = {}
    for _family, _fails in _state.get("failures", {}).items():
        if _fails:
            _failed = set(_fails)
            _state["done"][_family] = [p for p in _state["done"].get(_family, []) if p not in _failed]
            _state["progress"][_family] = len(_state["done"][_family])
            _state["failures"][_family] = {}
            _requeued[_family] = len(_failed)
    if _requeued:
        pv.atomic_json_dump(_state, state_path)
        print("Re-queued failed proteins:", _requeued)
    else:
        print("No failed proteins from earlier runs.")
else:
    print("Fresh run: no previous workspace.")
display(show_family_state())

Fresh run: no previous workspace.


,display_model,completed_proteins,total_proteins
0,ElasticNet,0,5113
1,XGBoost,0,5113


## 3. Run / resume ElasticNet only

Run this cell independently. It evaluates only the prespecified z-score, \(k=5000\) ElasticNet inventory. XGBoost is not evaluated in this cell.

In [10]:
enet_progress = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=VERIFY_NORMALIZERS,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=RESET_ALL,
    cleanup_work_on_success=False,
    model_families=("ElasticNet",),
    show_progress=True,
    finalize=False,
)
RESET_ALL = False  # Never erase the completed ENet run when the XGB cell is executed next.
gc.collect()
display(enet_progress)
display(show_family_state())

ElasticNet:   0%|          | 0/5113 [00:00<?, ?protein/s]

,display_model,completed_proteins,total_proteins,failures
0,ElasticNet,5113,5113,0
1,XGBoost,0,5113,0


,display_model,completed_proteins,total_proteins,percent
0,ElasticNet,5113,5113,100.0
1,XGBoost,0,5113,0.0


## 4. Run / resume XGBoost only

This cell uses the same disk-backed RNA inputs and workspace but evaluates **only XGBoost**. It can be executed later or resumed independently after interruption.

In [11]:
xgb_progress = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=VERIFY_NORMALIZERS,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=False,
    cleanup_work_on_success=False,
    model_families=("XGBoost",),
    show_progress=True,
    finalize=False,
)
gc.collect()
display(xgb_progress)
display(show_family_state())

XGBoost:   0%|          | 0/5113 [00:00<?, ?protein/s]

,display_model,completed_proteins,total_proteins,failures
0,ElasticNet,5113,5113,0
1,XGBoost,5113,5113,0


,display_model,completed_proteins,total_proteins,percent
0,ElasticNet,5113,5113,100.0
1,XGBoost,5113,5113,100.0


## 5. Finalize the 20 cohort checkpoints

This cell does **not** rerun either model family. It first requires both progress counters to equal their complete model inventories, then writes the per-cohort NPZ prediction checkpoints, compact metric partitions, transformation audits, and completion-status files consumed by notebooks 08–10.

In [12]:
completion = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=VERIFY_NORMALIZERS,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=False,
    cleanup_work_on_success=CLEANUP_WORK_AFTER_FINALIZE,
    model_families=(),
    show_progress=False,
    finalize=True,
)
completion_path = LAYOUT["evaluation"] / "evaluation_completion.csv"
completion.to_csv(completion_path, index=False)
display(completion)

  finalized BRAIN: 71 samples
  finalized BRAIN_2021: 108 samples
  finalized BREAST2020: 122 samples
  finalized BREAST_2016: 77 samples
  finalized COLON: 86 samples
  finalized COLON_2019: 95 samples
  finalized ENDOMET_VAL: 115 samples
  finalized HEADNECK_2021: 151 samples
  finalized LIVER: 62 samples
  finalized LIVER_VAL: 318 samples
  finalized LUNG: 76 samples
  finalized LUNG2021: 202 samples
  finalized LUNG_2020_SMOKE: 89 samples
  finalized LUNG_VAL: 211 samples
  finalized OVARY_2016: 119 samples
  finalized PANCREA2021: 140 samples
  finalized PEDBRAIN_2020: 188 samples
  finalized PROSTATE: 65 samples
  finalized RENAL_2019: 185 samples
  finalized STOMACH_2019: 80 samples


,cohort,completed_at,manifest_hash,metric_path,metric_sha256,model_failures,models,pipeline_version,platform,prediction_path,prediction_sha256,proteins,samples,scenarios,tmt_scale_sensitivity,xu_shortname
0,BRAIN,2026-11-04T14:08:27,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,3817218a08eb8e902b70fc4dcb6c30d36bbfe83dff26cb...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,LabelFree,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,85bb6a0dcc92f3c47c69e556ff319f2ea07a12e53e5634...,5113,71,"[""primary_raw"", ""clipped"", ""scale_aligned""]",not_applicable,BN_labelfree
1,BRAIN_2021,2026-11-04T14:08:35,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,676175aabd4013044b2eaaa1dfc044bee5769ff768736c...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,TMT,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,7b5f58a39115eeaffc4dc36b8936e7754355e9514eec32...,5113,108,"[""primary_raw"", ""clipped"", ""scale_aligned""]",TMT1,BN_tmt
2,BREAST2020,2026-11-04T14:08:43,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,d2186239515dc77ffb95600768d9a831b5be4741821286...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,TMT,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,763269123db16e5fd546d18251371e3c99f0e88748ff4a...,5113,122,"[""primary_raw"", ""clipped"", ""scale_aligned""]",TMT1,BR_tmt
3,BREAST_2016,2026-11-04T14:08:53,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,7cfeda2218c2b3a98c1daa7ce2e0d7d9f576aecf347294...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,iTRAQ,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,95d26e62cad211c978665203b5dd6ede0218fd6b2debd1...,5113,77,"[""primary_raw"", ""clipped"", ""scale_aligned""]",not_applicable,BR_itraq
4,COLON,2026-11-04T14:08:58,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,1122c525f88557b496641339cfd4a4053aaf1ca116bf31...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,LabelFree,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,87b0dcf0fbf3a616aa3d07784cc769fd9820cdc5c67da2...,5113,86,"[""primary_raw"", ""clipped"", ""scale_aligned""]",not_applicable,CO_labelfree
5,COLON_2019,2026-11-04T14:09:05,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,4a2f42005c8178170cf293048e28051ca22c28b8958ea3...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,TMT,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,484650e446be8bcf3991360e1332cf75102dfee581457e...,5113,95,"[""primary_raw"", ""clipped"", ""scale_aligned""]",TMT1,CO_tmt
6,ENDOMET_VAL,2026-11-04T14:09:12,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,37e653ef936c967a722e42f79b04f4528a88e02b8c80be...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,TMT,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,4c470e8bc749e6f93d3b1a74fb562ff080f5627844a9de...,5113,115,"[""primary_raw"", ""clipped"", ""scale_aligned""]",TMT1,EC_tmt
7,HEADNECK_2021,2026-11-04T14:09:24,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,b3882709eece2e59309438f8497aac091535233e092f7d...,0,"[""RawProxy"", ""ZscoreProxy"", ""LinearProxy"", ""El...",pRNA-external-validation-1.0,TMT,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,83b2c6be128ee353d36f47dcbb62322398d11e3f237dee...,5113,151,"[""primary_raw"", ""clipped"", ""scale_aligned""]",TMT2,HN_tmt
8,LIVER,2026-11-04T14:09:30,98cdc5451b903c1769428a4757a2859815a411310b36ef...,C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-re...,5f5cac1b3055dd1d21

## Compact same-gene proxy table for notebook 8

Notebook 8 compares the three CCLE-trained same-gene proxies (raw RNA, CCLE-train z-score, CCLE-train linear regression) per cohort and protein, and keeps the best one per protein, cohort and metric (paper: *Input transformation and external validation*).
This cell writes them from the metric partitions just finalised, so no separate proxy notebook is needed.

Output: `05. validation/data/proxy_models/proxy_metrics_by_tissue_protein.csv`, the default path notebook 8 reads.

In [13]:
PROXY_TABLE_DIR = PROJECT_ROOT / "data" / "proxy_models"
PROXY_TABLE_DIR.mkdir(parents=True, exist_ok=True)
proxy_tissue_protein = pv.proxy_metrics_by_tissue_protein(LAYOUT["metric_partitions"], MANIFEST_HASH)
proxy_tissue_protein_path = PROXY_TABLE_DIR / "proxy_metrics_by_tissue_protein.csv"
proxy_tissue_protein.to_csv(proxy_tissue_protein_path, index=False)
coverage = proxy_tissue_protein.groupby("display_model")["tissue"].nunique()
if (coverage != len(pv.COHORT_TABLE)).any():
    raise RuntimeError(f"Proxy table does not cover all cohorts for every proxy variant:\n{coverage}")
print("Wrote", proxy_tissue_protein_path, f"({len(proxy_tissue_protein):,} rows)")
display(coverage.rename("cohorts").to_frame())

Wrote C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\data\proxy_models\proxy_metrics_by_tissue_protein.csv (306,780 rows)


,cohorts
display_model,
LinearProxy,20
RawProxy,20
ZscoreProxy,20


## 6. Final completeness and provenance audit

In [14]:
from collections import Counter

# Report any model failures with their reasons before the final checks
if state_path.exists():
    _state = json.loads(state_path.read_text(encoding="utf-8"))
    for _family, _fails in _state.get("failures", {}).items():
        if _fails:
            print(f"{_family}: {len(_fails):,} failed")
            for _msg, _n in Counter(m[:120] for m in _fails.values()).most_common(5):
                print(f"   {_n:6,} x {_msg}")

completion = pv.validate_evaluation_completion(
    LAYOUT["evaluation_status"],
    MANIFEST_HASH,
    pv.COHORT_TABLE["cohort"].tolist(),
)
if len(completion) != 20:
    raise RuntimeError(f"Expected 20 completed cohorts, found {len(completion)}")
if completion["model_failures"].sum() != 0:
    raise RuntimeError("One or more primary model predictions failed - see the reasons printed above.")

expected_models = {"LinearProxy", "ElasticNet", "XGBoost"}
for models in completion["models"].astype(str):
    # Stored JSON lists may render as strings in the completion DataFrame.
    if not all(name in models for name in expected_models):
        raise RuntimeError(f"Incomplete finalized model set: {models}")

provenance = pv.notebook_provenance(
    "07_evaluation_paper_ready.ipynb",
    MANIFEST_HASH,
    {
        "manifest": str(MANIFEST_PATH.resolve()),
        "model_inventory_hash": manifest["model_inventory_hash"],
        "execution_mode": "separate_enet_xgb_progress_bars",
    },
    {
        "completion_table": str(completion_path.resolve()),
        "prediction_directory": str(LAYOUT["prediction_checkpoints"].resolve()),
        "metric_directory": str(LAYOUT["metric_partitions"].resolve()),
    },
)
pv.atomic_json_dump(provenance, LAYOUT["evaluation"] / "notebook07_provenance.json")

print("Completed cohorts:", len(completion))
print("ElasticNet run separately:", True)
print("XGBoost run separately:", True)
print("Progress bars:", "tqdm protein-wise")
print("Prediction checkpoints use long rows:", False)
print("External scTranslator:", "removed")

Completed cohorts: 20
ElasticNet run separately: True
XGBoost run separately: True
Progress bars: tqdm protein-wise
Prediction checkpoints use long rows: False
External scTranslator: removed
